## 01 - Add Gemini Key and Check Token Usage

# 🧠 Thinking Level in Google Gemini 3 API

## What “Thinking” means in Gemini

- **Internal reasoning process**  
  Used by Gemini models to improve quality on complex tasks.  
  *“Thinking” involves reasoning, planning, and multi-step logic.*

- **Part of the model architecture**  
  The model *thinks through* the prompt internally before producing the answer.

- **Not just text completion**  
  Can involve multi-step internal computation rather than simple token prediction.

- **Model limitation note**  
  Gemini 3 Pro or Gemini 3 Flash **does not support full thinking-off**.

---

## Thinking Level Support by Model

| Model            | `thinking_level` parameter support |
|------------------|------------------------------------|
| **Gemini 3 Pro** | `low` — minimal reasoning; lower latency & cost  
|                  | `high` — deeper reasoning; better quality for complex queries |
| **Gemini 3 Flash** | `low` & `high`  
|                  | `medium` — balanced thinking  
|                  | `minimal` — closest to “no thinking” for most use cases |

---


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3-pro-preview", contents="What is Agentic AI?",
)

prompt_tokens = response.usage_metadata.prompt_token_count
candidate_tokens = response.usage_metadata.candidates_token_count
total_tokens = response.usage_metadata.total_token_count

print(f"Prompt Tokens: {prompt_tokens}")
print(f"Response Tokens: {candidate_tokens}")
print(f"Total Tokens: {total_tokens}")

# Extract specific metadata for "Thinking" (if applicable)
thought_tokens = response.usage_metadata.thoughts_token_count
if thought_tokens:
    print(f"Thinking Tokens: {thought_tokens}")
    
print(response.text)

## 02 - Print Gemini 3 Thinking Level Details

In [33]:
from google import genai
from google.genai import types

client = genai.Client()

# prompt = """
# Three servers are sitting side-by-side in a rack: Server A, Server B, and Server C. 
# Each runs a different OS: Linux, Windows, and BSD.

# 1. The Linux server is in the Middle.
# 2. Server A is to the left of the Windows server.
# 3. Server C is not the Linux server.

# Which OS is running on Server A, Server B, and Server C?
# """

prompt = "how many times the character 'e' repeats in this sentence"
thoughts = ""
answer = ""

for chunk in client.models.generate_content_stream(
    model="gemini-3-flash-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
      thinking_config=types.ThinkingConfig(
        include_thoughts=True,
        thinking_level="minimal"
      )
    )
):
  for part in chunk.candidates[0].content.parts:
    if not part.text:
      continue
    elif part.thought:
      if not thoughts:
        print("Thoughts summary:")
      print(part.text)
      thoughts += part.text
    else:
      if not answer:
        print("Answer:")
      print(part.text)
      answer += part.text
  



Answer:
There are **6** instances of the character 'e' in that sentence:

1
. how many tim**e**s (1)
2. th**e** (2)
3. charact**
e**r (3)
4. '**e**' (4)
5. r**e**p**e
**ats (5, 6)


```mermaid
flowchart LR
    subgraph A["Answer for the reference"]
        A1["A - BSD"]
        A2["B - Linux"]
        A3["C - Windows"]
    end
```

## 03 Final Gemini 3-pro code with thinking_level

In [27]:
from google import genai
from google.genai import types

client = genai.Client()

# prompt = """
# Three servers are sitting side-by-side in a rack: Server A, Server B, and Server C. 
# Each runs a different OS: Linux, Windows, and BSD.

# 1. The Linux server is in the Middle.
# 2. Server A is to the left of the Windows server.
# 3. Server C is not the Linux server.

# Which OS is running on Server A, Server B, and Server C?
# """

prompt = "how many times character 'e' repeats in this sentence"

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(include_thoughts=True, thinking_level="minimal"))
)

prompt_tokens = response.usage_metadata.prompt_token_count
candidate_tokens = response.usage_metadata.candidates_token_count
total_tokens = response.usage_metadata.total_token_count


print(f"Prompt Tokens: {prompt_tokens}")
print(f"Response Tokens: {candidate_tokens}")
print(f"Total Tokens: {total_tokens}")

# Extract specific metadata for "Thinking" (if applicable)
thought_tokens = response.usage_metadata.thoughts_token_count
if thought_tokens:
    print(f"Thinking Tokens: {thought_tokens}")

print("--- MODEL THOUGHTS ---")
for part in response.candidates[0].content.parts:
    # In the SDK, thought parts have a 'thought' attribute set to True
    if part.thought:
        print(part.text)

print("\n--- FINAL RESPONSE ---")
print(response.text)
'''
High
----
Prompt Tokens: 12
Response Tokens: 88
Total Tokens: 1385
Thinking Tokens: 1285

low
---
Prompt Tokens: 12
Response Tokens: 62
Total Tokens: 177
Thinking Tokens: 103


'''

Prompt Tokens: 12
Response Tokens: 231
Total Tokens: 243
--- MODEL THOUGHTS ---

--- FINAL RESPONSE ---
In the sentence **"how many times character 'e' repeats in this sentence"**, the character 'e' repeats **7** times:

1. how r**e**p**e**ats (1 & 2)
2. tim**e**s (3)
3. charact**e**r (4)
4. r**e**p**e**ats (5 & 6)
5. s**e**nt**e**nc**e** (7, 8, 9)

Wait, let's look closer at the specific sentence you provided: **"how many times character 'e' repeats in this sentence"**

1. tim**e**s (1)
2. charact**e**r (2)
3. '**e**' (3)
4. r**e**p**e**ats (4, 5)
5. s**e**nt**e**nc**e** (6, 7, 8)

There are **8** occurrences of the letter 'e'.


'\nHigh\n----\nPrompt Tokens: 12\nResponse Tokens: 88\nTotal Tokens: 1385\nThinking Tokens: 1285\n\nlow\n---\nPrompt Tokens: 12\nResponse Tokens: 62\nTotal Tokens: 177\nThinking Tokens: 103\n\n\n'

## 04 Gemini 2.5  - thinking_budget

In [30]:
from google import genai
from google.genai import types

client = genai.Client()

# prompt = """
# Three servers are sitting side-by-side in a rack: Server A, Server B, and Server C. 
# Each runs a different OS: Linux, Windows, and BSD.

# 1. The Linux server is in the Middle.
# 2. Server A is to the left of the Windows server.
# 3. Server C is not the Linux server.

# Which OS is running on Server A, Server B, and Server C?
# """

prompt = "how many times character 'e' repeats in this sentence"

thoughts = ""
answer = ""

for chunk in client.models.generate_content_stream(
    model="gemini-2.5-pro",
    contents=prompt,
    config=types.GenerateContentConfig(
      thinking_config=types.ThinkingConfig(
        include_thoughts=True,
        thinking_budget=0
      )
    )
):
  for part in chunk.candidates[0].content.parts:
    if not part.text:
      continue
    elif part.thought:
      if not thoughts:
        print("Thoughts summary:")
      print(part.text)
      thoughts += part.text
    else:
      if not answer:
        print("Answer:")
      print(part.text)
      answer += part.text

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Budget 0 is invalid. This model only works in thinking mode.', 'status': 'INVALID_ARGUMENT'}}

## 05 Final Gemini 2-5 Code with thinking_budget

In [31]:
from google import genai
from google.genai import types

client = genai.Client()

# prompt = """
# Three servers are sitting side-by-side in a rack: Server A, Server B, and Server C. 
# Each runs a different OS: Linux, Windows, and BSD.

# 1. The Linux server is in the Middle.
# 2. Server A is to the left of the Windows server.
# 3. Server C is not the Linux server.

# Which OS is running on Server A, Server B, and Server C?
# """

prompt = "how many times character 'e' repeats in this sentence"

response = client.models.generate_content(
    model="gemini-2.5-pro",
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(include_thoughts=True, thinking_budget=1024))
)

prompt_tokens = response.usage_metadata.prompt_token_count
candidate_tokens = response.usage_metadata.candidates_token_count
total_tokens = response.usage_metadata.total_token_count


print(f"Prompt Tokens: {prompt_tokens}")
print(f"Response Tokens: {candidate_tokens}")
print(f"Total Tokens: {total_tokens}")

# 2. Extract specific metadata for "Thinking" (if applicable)
thought_tokens = response.usage_metadata.thoughts_token_count
if thought_tokens:
    print(f"Thinking Tokens: {thought_tokens}")

print("--- MODEL THOUGHTS ---")
for part in response.candidates[0].content.parts:
    # In the SDK, thought parts have a 'thought' attribute set to True
    if part.thought:
        print(part.text)

print("\n--- FINAL RESPONSE ---")
print(response.text)

Prompt Tokens: 12
Response Tokens: 89
Total Tokens: 555
Thinking Tokens: 454
--- MODEL THOUGHTS ---
**Calculating 'e' Occurrences in a Sentence**

Okay, so the user wants the count of the letter 'e' in that particular sentence, a straightforward task. First, let's identify the sentence itself: "how many times character 'e' repeats in this sentence."

Now, to the count. I'll methodically go through each word, keeping track.

*   "how" - Zero 'e's.
*   "many" - Still zero.
*   "times" - Aha, one!
*   "character" - Two here!
*   "'e'" - Another one, right in the quoted letter!
*   "repeats" - Two more.
*   "in" - Nothing.
*   "this" - Nope.
*   "sentence" - And finally, three more 'e's.

Adding it all up: 1 (times) + 2 (character) + 1 ('e') + 2 (repeats) + 3 (sentence) equals 9.

Therefore, the character 'e' appears 9 times. I'll make sure to provide a clear answer, and maybe I'll even show the breakdown for extra clarity:

*   sent**e**nc**e** (3)
*   charact**e**r (2)
*   r**e**p**e**at